# 第 7 周 - 笔记本 4：评估

## 目标
评估微调模型：
1. 从 HuggingFace Hub 加载微调后的模型
2. 在测试集上运行
3. 生成可视化
4. 与基线比较

预期结果：约 40 美元的误差（相对于 110 美元的基线）

## 时间：15-20 分钟

In [ ]:
import sys
sys.path.append('..')

import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

from src.items import Item
from src.evaluator import evaluate
from src.config import config

# 负载环境
# Load environment
load_dotenv()
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")
print(f"GPU available: {torch.cuda.is_available()}")

## 配置

In [ ]:
# 评估模型
# Model to evaluate
FINETUNED_MODEL_ID = "your-username/llama-pricer-lite"  # Replace with your model

print(f"Will evaluate model: {FINETUNED_MODEL_ID}")
config.display()

## 加载测试数据

In [ ]:
print(f"Loading test data from: {config.DATASET_NAME}")
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")

## 加载微调模型

In [ ]:
# 配置 4 位量化
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {config.BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading LoRA adapters from: {FINETUNED_MODEL_ID}")
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_ID)

# 设置填充标记
# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("✅ Fine-tuned model loaded")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 对样品产品进行测试

In [ ]:
def predict_finetuned(item: Item) -> str:
    """
    Predict price using fine-tuned model
    """
    # 创建提示
    # Create prompt
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)
    
    prompt = item.test_prompt()
    
    # 标记化
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 产生
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # 解码
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 提取完成
    # Extract completion
    completion = response.split(config.PREFIX)[-1].strip()
    
    return completion

In [ ]:
# 测试几个例子
# Test on a few examples
print("Testing fine-tuned model on 5 sample products:\n")

for i in range(5):
    item = test[i]
    prediction = predict_finetuned(item)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)

## 对测试集的全面评估

In [ ]:
# 在完整的测试集上进行评估
# Evaluate on full test set
results = evaluate(
    predict_finetuned,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

print(f"\n{'='*60}")
print("FINE-TUNED MODEL RESULTS")
print(f"{'='*60}")
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")

## 与基线比较

In [ ]:
import plotly.graph_objects as go

# 预期结果（根据实际结果更新）
# Expected results (update with your actual results)
baseline_error = 110.72  # Base Llama
finetuned_error = results['average_error']
improvement = (baseline_error - finetuned_error) / baseline_error * 100

# 创建比较图表
# Create comparison chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=["Base Llama 3.2", "Fine-tuned Llama"],
    y=[baseline_error, finetuned_error],
    marker_color=["darkred", "green"],
    text=[f"${baseline_error:.2f}", f"${finetuned_error:.2f}"],
    textposition="outside",
))

fig.update_layout(
    title=f"Fine-tuning Improvement: {improvement:.1f}% reduction in error",
    yaxis_title="Mean Absolute Error ($)",
    width=800,
    height=500,
)

fig.show()

print(f"\n🎉 Fine-tuning reduced error by {improvement:.1f}%!")
print(f"   From ${baseline_error:.2f} → ${finetuned_error:.2f}")

## 与所有型号比较

In [ ]:
# 所有模型结果（来自第 7 周课程）
# All model results (from Week 7 curriculum)
all_results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4-bit", "darkred", 110.72),
    ("Fine-tuned (Your Model)", "green", finetuned_error),
]

labels, colors, values = zip(*all_results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="All Models Comparison - Price Prediction Error",
    yaxis=dict(range=[0, max(values)], title="Mean Absolute Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1200,
    height=600,
)

fig.show()

## 概括

✅ 评估完成！

**您的结果：**
- 微调模型：$XX.XX 错误
- 基础改善：XX%
- 与 GPT-5.1 的比较：[更好/更差]$XX

**预期结果：**
- Lite 模式：~$65 错误（好）
- 完整模式：大约 40 美元的错误（非常好 - 击败 GPT-5.1！）

**主要成就：**
1. ✅ 成功使用 QLoRA 微调 Llama 3.2
2. ✅ 与基本模型相比，误差减少了 60-70%
3. ✅ 实现比前沿型号有竞争力/更好的性能
4. ✅ 无 API 成本 - 可以在本地或廉价的 GPU 上运行
5. ✅ 完全控制模型和数据

**后续步骤：**
- 部署到生产环境（Modal.com、HuggingFace 端点等）
- 尝试先进的技术（多域适配器、推理链）
- 多代理系统移至第 8 周
- 结合第 7 周 + 第 8 周形成完整的平台